# iCanClean — and why a negative control is mandatory

When you record dedicated noise-reference channels, CCA can project out the subspace they share with the scalp. The catch is that CCA will always find *some* shared subspace — so attenuation alone proves nothing.

*Deep dive behind the [five-minute demo](../meta_mne_denoise_demo.ipynb).*

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("demo_utils.py").exists():
    done = subprocess.run(
        ["git", "clone", "-q", "--depth", "1",
         "https://github.com/snesmaeili/mne-denoise-meta-demo.git"],
        capture_output=True, text=True)
    if done.returncode != 0:
        raise SystemExit(
            "Could not clone the demo repository. If it is still private, the Colab "
            "VM has no credentials for it -- authorising Colab lets it OPEN a "
            "notebook, not clone the repo. Make it public, or run locally."
        )
    os.chdir("mne-denoise-meta-demo")
    sys.path.insert(0, os.getcwd())
try:
    import mne_denoise  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mne-denoise @ git+https://github.com/mne-tools/mne-denoise.git@f5b821cc2a535e84ed46085d45ea5a356dd8d548"],
                   check=True)

import warnings, logging
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level("ERROR")
logging.getLogger("mne_denoise").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*Epochs are not baseline corrected.*")
%matplotlib inline
RANDOM_STATE = 97

## Scalp + reference, with a genuinely shared artifact

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sfreq, n_eeg, n_ref, dur = 250.0, 32, 8, 120.0
n = int(sfreq * dur)

brain = rng.standard_normal((n_eeg, n)) * 1e-5
artifact = rng.standard_normal((4, n)) * 3e-5          # the shared source
A_eeg = rng.standard_normal((n_eeg, 4))
A_ref = rng.standard_normal((n_ref, 4))
eeg = brain + A_eeg @ artifact
ref = A_ref @ artifact + rng.standard_normal((n_ref, n)) * 5e-6

names = [f"EEG{i:03d}" for i in range(n_eeg)] + [f"N-{i:03d}" for i in range(n_ref)]
info = mne.create_info(names, sfreq, "eeg")
raw = mne.io.RawArray(np.vstack([eeg, ref]), info, verbose="ERROR")
ref_names = [c for c in raw.ch_names if c.startswith("N-")]
print(f"{n_eeg} scalp + {n_ref} reference channels, {dur:.0f} s")

## The method, and the control it must beat

The control feeds iCanClean the *same* reference channels, circularly shifted in time. Same spectra, no true alignment. Whatever it removes there, it removes by overfitting.

In [ ]:
from mne_denoise.icanclean import ICanClean

def run(data, label):
    r = mne.io.RawArray(data, info, verbose="ERROR")
    est = ICanClean(sfreq=sfreq, ref_channels=ref_names,
                    primary_channels=[c for c in raw.ch_names if c.startswith("EEG")])
    out = est.fit_transform(r)
    cleaned = out.get_data()[:n_eeg]
    kept = np.var(cleaned) / np.var(eeg)
    err = np.linalg.norm(cleaned - brain) / np.linalg.norm(brain)
    print(f"  {label:24s} removed {est.n_removed_.mean():5.2f} comp/win  "
          f"variance kept {kept:.3f}  error-vs-brain {err:.3f}")
    return est

full = np.vstack([eeg, ref])
shifted = np.vstack([eeg, np.roll(ref, int(37.0 * sfreq), axis=1)])

print(f"  {'uncorrected':24s} {'':30s} error-vs-brain "
      f"{np.linalg.norm(eeg - brain) / np.linalg.norm(brain):.3f}")
est_real = run(full, "real reference")
est_ctrl = run(shifted, "time-shifted reference")

> If those two lines are close, the attenuation is not evidence that a real shared artifact was found. On 120-channel mobile EEG we measured 11.5% vs 10.5% — indistinguishable. Always run the control.

## Canonical correlations tell you which it was

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for est, label, colour in [(est_real, "real reference", "#009E73"),
                           (est_ctrl, "time-shifted", "#D55E00")]:
    ax.plot(np.sort(est.correlations_.mean(0))[::-1], "o-", label=label, color=colour)
ax.set_xlabel("component"); ax.set_ylabel("mean squared canonical correlation")
ax.set_title("A real shared subspace separates from its own null")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.show()